:# Build a Retrieval Augmented Generation (RAG) System

## What is RAG?
RAG combines:
- **Retrieval**: Finding relevant documents from a knowledge base
- **Generation**: Using an LLM to generate answers based on retrieved context

## What we'll build:
A system that:
1. Loads a dataset from Hugging Face
2. Splits documents into chunks
3. Creates embeddings for semantic search
4. Stores embeddings in a FAISS vector database
5. Retrieves relevant documents based on queries
6. Uses an LLM to generate answers from retrieved context

Let's get started! 🚀

In [23]:
# Install all required libraries
!pip install -q langchain==0.0.350
!pip install -q transformers==4.35.0
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu
!pip install -q torch

print("✅ All libraries installed successfully!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.4.1 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 0.17.3 which is incompatible.
peft 0.17.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.17.3 which is incompatible.
sentence-transformers 5.1.2 requires huggingface-hub>=0.20.0, but you have huggingface-hub 0.17.3 which is incompatible.
sentence-transformers 5.1.2 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.35.0 which is incompatible.
accelerate 1.11.0 requires huggingface_hub>=0.21.0, but you have huggingface-hub 0.17.3 which is incompatible.
diffusers 0.35.2 requires huggingface-hub>=0.34.0, but you have huggingface-hub 0.17.3 which is incompatible.
gradio 5.49.1 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.17.3 which is incompatible.
gradio-client 1.13.3 requires h

In [24]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("Running on CPU")

PyTorch version: 2.8.0+cu126
CUDA available: True
CUDA device: Tesla T4
Number of GPUs: 1


In [25]:
from langchain.document_loaders import HuggingFaceDatasetLoader

# Specify dataset name and content column
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

# Load the dataset
print(f"Loading dataset: {dataset_name}")
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

# Verify loading
print(f"\n✅ Dataset loaded successfully!")
print(f"Total documents: {len(data)}")
print("\n" + "="*80)
print("First document preview:")
print("="*80)
print(f"Content: {data[0].page_content[:300]}...")
print(f"Metadata: {data[0].metadata}")

Loading dataset: databricks/databricks-dolly-15k

✅ Dataset loaded successfully!
Total documents: 15011

First document preview:
Content: "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major ...
Metadata: {'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [26]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

# Split documents
print("Splitting documents into chunks...")
docs = text_splitter.split_documents(data)

print(f"\n✅ Documents split successfully!")
print(f"Total chunks created: {len(docs)}")
print("\n" + "="*80)
print("First chunk preview:")
print("="*80)
print(docs[0])

Splitting documents into chunks...

✅ Documents split successfully!
Total chunks created: 18502

First chunk preview:
page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [28]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🔥 Using device: {device}")

# Define model path and configurations
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device': device}  # ← CHANGE ICI : utilise le GPU !
encode_kwargs = {'normalize_embeddings': False}

# Initialize embeddings model
print(f"Loading embedding model: {modelPath}")
embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

print("✅ Embedding model loaded successfully on GPU!")

# Test embedding creation
test_text = "This is a test document for embedding."
query_result = embeddings.embed_query(test_text)

print(f"\n📊 Embedding dimension: {len(query_result)}")
print(f"First 5 values: {query_result[:5]}")

🔥 Using device: cuda
Loading embedding model: sentence-transformers/all-MiniLM-l6-v2
✅ Embedding model loaded successfully on GPU!

📊 Embedding dimension: 384
First 5 values: [-0.06414405256509781, 0.02553531900048256, 0.0007875760202296078, 0.019725030288100243, 0.04553202912211418]


In [29]:
from langchain.vectorstores import FAISS

# Create FAISS vector store
print("Creating FAISS vector store...")
print("⚠️ This will take 2-5 minutes with GPU T4...")
print(f"Processing {len(docs)} chunks...")

db = FAISS.from_documents(docs, embeddings)

print("\n✅ FAISS vector store created successfully!")
print(f"Total vectors in store: {db.index.ntotal}")

Creating FAISS vector store...
⚠️ This will take 2-5 minutes with GPU T4...
Processing 18502 chunks...

✅ FAISS vector store created successfully!
Total vectors in store: 18502


In [37]:
from transformers import pipeline

# Verify FAISS database exists
print(f"✅ FAISS database ready with {db.index.ntotal} vectors\n")

# Create retriever from FAISS
retriever = db.as_retriever(search_kwargs={"k": 4})
print("✅ Retriever created (retrieving top 4 documents)\n")

# Create QA pipeline
print("Loading QA model: Intel/dynamic_tinybert")
qa_pipeline = pipeline(
    "question-answering",
    model="Intel/dynamic_tinybert",
    tokenizer="Intel/dynamic_tinybert"
)
print("✅ QA model loaded\n")

def answer_question(question):
    """
    Manual RAG implementation:
    1. Retrieve relevant documents
    2. Pass to QA model
    3. Return answer and confidence
    """
    # Step 1: Retrieve relevant documents
    docs_retrieved = retriever.get_relevant_documents(question)

    # Step 2: Combine contexts from retrieved documents
    context = " ".join([doc.page_content for doc in docs_retrieved])

    # Step 3: Limit context length (model max is 512 tokens)
    if len(context) > 2000:
        context = context[:2000]

    # Step 4: Run QA pipeline
    result = qa_pipeline(question=question, context=context)

    # Debug: print what we get
    print(f"DEBUG - Result type: {type(result)}")
    print(f"DEBUG - Result content: {result}")

    # Handle the result properly
    if isinstance(result, dict):
        return result
    elif isinstance(result, list) and len(result) > 0:
        return result[0]
    else:
        return {'answer': str(result), 'score': 0.0}

print("="*80)
print("🎉 RAG SYSTEM READY TO ANSWER QUESTIONS!")
print("="*80)

✅ FAISS database ready with 18502 vectors

✅ Retriever created (retrieving top 4 documents)

Loading QA model: Intel/dynamic_tinybert


Invalid model-index. Not loading eval results into CardData.
Device set to use cuda:0


✅ QA model loaded

🎉 RAG SYSTEM READY TO ANSWER QUESTIONS!


In [41]:
from transformers import pipeline

# Verify FAISS database exists
print(f"✅ FAISS database ready with {db.index.ntotal} vectors\n")

# Create retriever from FAISS
retriever = db.as_retriever(search_kwargs={"k": 4})
print("✅ Retriever created (retrieving top 4 documents)\n")

# Create QA pipeline
print("Loading QA model: Intel/dynamic_tinybert")
qa_pipeline = pipeline(
    "question-answering",
    model="Intel/dynamic_tinybert",
    tokenizer="Intel/dynamic_tinybert"
)
print("✅ QA model loaded\n")

def answer_question(question):
    """
    Manual RAG implementation:
    1. Retrieve relevant documents
    2. Pass to QA model
    3. Return answer and confidence
    """
    # Step 1: Retrieve relevant documents
    docs_retrieved = retriever.get_relevant_documents(question)

    # Step 2: Combine contexts from retrieved documents
    context = " ".join([doc.page_content for doc in docs_retrieved])

    # Step 3: Limit context length (model max is 512 tokens)
    if len(context) > 2000:
        context = context[:2000]

    # Step 4: Run QA pipeline
    result = qa_pipeline(question=question, context=context)

    # Handle different return types
    if isinstance(result, dict):
        return {
            'answer': result.get('answer', ''),
            'score': result.get('score', 0.0)
        }
    elif isinstance(result, list) and len(result) > 0:
        first = result[0]
        if isinstance(first, dict):
            return {
                'answer': first.get('answer', ''),
                'score': first.get('score', 0.0)
            }
        else:
            return {'answer': str(first), 'score': 0.0}
    else:
        # It's just a string
        return {'answer': str(result), 'score': 1.0}

print("="*80)
print("🎉 RAG SYSTEM READY TO ANSWER QUESTIONS!")
print("="*80)

✅ FAISS database ready with 18502 vectors

✅ Retriever created (retrieving top 4 documents)

Loading QA model: Intel/dynamic_tinybert


Invalid model-index. Not loading eval results into CardData.
Device set to use cuda:0


✅ QA model loaded

🎉 RAG SYSTEM READY TO ANSWER QUESTIONS!


In [43]:
# Test questions
test_questions = [
    "What is cheesemaking?",
    "How does machine learning work?",
    "What is artificial intelligence?"
]

print("🤖 TESTING RAG SYSTEM")
print("="*80)

for question in test_questions:
    print(f"\n❓ Question: {question}")
    print("-"*80)

    try:
        result = answer_question(question)
        print(f"💡 Answer: {result['answer']}")
        print(f"📊 Confidence: {result['score']:.4f}")
    except Exception as e:
        print(f"⚠️ Error: {str(e)}")

    print("="*80)

🤖 TESTING RAG SYSTEM

❓ Question: What is cheesemaking?
--------------------------------------------------------------------------------
💡 Answer: to control the spoiling of milk into cheese
📊 Confidence: 0.4095

❓ Question: How does machine learning work?
--------------------------------------------------------------------------------
💡 Answer: methods that leverage data to improve performance on some set of tasks
📊 Confidence: 0.1564

❓ Question: What is artificial intelligence?
--------------------------------------------------------------------------------
💡 Answer: intelligence demonstrated by machines
📊 Confidence: 0.4199


In [44]:
def ask_rag(question):
    """
    Simple function to ask questions to the RAG system

    Args:
        question: Your question as a string

    Returns:
        Prints the answer and confidence score
    """
    try:
        result = answer_question(question)
        print(f"\n{'='*80}")
        print(f"Question: {question}")
        print(f"{'='*80}")
        print(f"Answer: {result['answer']}")
        print(f"Confidence: {result['score']:.4f}")
        print(f"{'='*80}\n")
        return result['answer']
    except Exception as e:
        print(f"Error: {str(e)}")
        return None

# Example usage
ask_rag("What is cheesemaking?")


Question: What is cheesemaking?
Answer: to control the spoiling of milk into cheese
Confidence: 0.4095



'to control the spoiling of milk into cheese'

## RAG System Summary

### What We Built:
1. ✅ **Dataset Loading**: Loaded databricks-dolly-15k dataset with 15,000 instruction-response pairs
2. ✅ **Document Chunking**: Split documents into manageable chunks (1000 chars with 150 overlap)
3. ✅ **Embeddings**: Created semantic embeddings using sentence-transformers
4. ✅ **Vector Store**: Indexed embeddings in FAISS for fast similarity search
5. ✅ **LLM Integration**: Used Intel/dynamic_tinybert for question answering
6. ✅ **QA Chain**: Built a complete RAG pipeline with Langchain

### How It Works:
```
User Question → Embed Query → FAISS Retrieval → Get Top-K Documents → LLM Answer Generation
```

### Key Components:
- **Retriever**: Finds top 4 most relevant documents using cosine similarity
- **LLM**: Generates answers based on retrieved context
- **Chain Type**: "refine" - iteratively improves answer across retrieved docs

### Benefits of RAG:
- ✅ Grounded answers (based on actual documents)
- ✅ Reduced hallucinations
- ✅ Updatable knowledge (just update the vector store)
- ✅ Source attribution possible
- ✅ Works with smaller LLMs

### Potential Improvements:
- Use larger/better embedding models (e.g., all-mpnet-base-v2)
- Increase chunk overlap for better context preservation
- Experiment with different retriever k values
- Try different LLM models
- Add source document tracking
- Implement re-ranking for better retrieval